In [1]:
import sys
import os

# Add the project root directory to Python's path
# This allows us to import 'qgcn_lib' from inside 'examples/notebooks'
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)



In [2]:
import torch
from qgcn_lib.datasets import MicroBenchmark

# Generate a synthetic graph
# - n_nodes=256: Small enough for rapid CPU simulation.
# - d_feat=16: Chosen specifically because log2(16) = 4 qubits.
# - n_clusters=3: Ground truth communities for NMI evaluation.
dataset = MicroBenchmark(root='./data/micro', n_nodes=256, d_feat=16, n_clusters=3)
data = dataset[0]



print(f"Graph Created: {data.num_nodes} Nodes, {data.num_features} Features")
# Output: Graph Created: 256 Nodes, 16 Features
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Generating MicroBenchmark data...
Nodes: 256 | Edges: 4800 | Qubit Req: 4
Graph Created: 256 Nodes, 16 Features


Processing...
Done!


In [3]:
data

Data(x=[256, 16], edge_index=[2, 4800], y=[256])

In [6]:
import math
from qgcn_lib.nn import SummaryMLP, NISQQGCNConv
from qgcn_lib.utils import set_all_seeds, feature_shuffling_corruption
from torch_geometric.nn import DeepGraphInfomax
import tqdm


# Automatic Qubit Calculation
# For our micro-benchmark (d=16), this results in exactly 4 qubits.
in_channels = data.x.size(1)
hidden_channels = math.ceil(math.log2(in_channels))

print(f"--> Feature Dimension (d): {in_channels}")
print(f"--> Quantum Latent Space: {hidden_channels} Qubits")

def train_quantum_dgi(model, features, edge_index, epochs):
    """
    Training loop with Group-Wise Learning Rates.
    """
    model.to(device)
    features = features.to(device)
    edge_index = edge_index.to(device)

    param_groups = []
    
   
    if hasattr(model.encoder, 'qc'):
        param_groups.append({'params': model.encoder.qc.parameters(), 'lr': 0.01})
    if hasattr(model.encoder, 'local_mp'):
        param_groups.append({'params': model.encoder.local_mp.parameters(), 'lr': 0.01})

    classical_params = []
    if hasattr(model.encoder, 'q_proj'): classical_params.extend(model.encoder.q_proj.parameters())
    if hasattr(model.encoder, 'bias'): classical_params.append(model.encoder.bias)
    if hasattr(model.encoder, 'prelu'): classical_params.extend(model.encoder.prelu.parameters())
    if hasattr(model, 'summary'): classical_params.extend(model.summary.parameters())
    
    param_groups.append({'params': classical_params, 'lr': 0.001})

    # Initialize Optimizer with the grouped parameters
    optimizer = torch.optim.Adam(param_groups)

    # Standard DGI Training Loop
    print(f"--> Starting Training for {epochs} epochs...")
    for epoch in tqdm.tqdm(range(epochs), desc="Training"):
        model.train()
        optimizer.zero_grad()
        
        # Forward pass returns: Real Embeddings, Corrupted Embeddings, Global Summary
        pos_z, neg_z, summary = model(features, edge_index)
        
        loss = model.loss(pos_z, neg_z, summary)
        loss.backward()
        optimizer.step()
        
    return model

def run_experiment(features, idx_edge):
    # 1. Reproducibility
    # Essential for quantum simulations where measurement outcomes can be stochastic.
    set_all_seeds(123)
    
   
    
    # 2. Initialize the Quantum Encoder
    encoder = NISQQGCNConv(
        in_channels=features.size(1),  
        q_depth=3 
    )
    
    # 3. Initialize the Global Summary Network
    summary = SummaryMLP(hidden_channels)
    
    # 4. Construct the DGI Model
    # We use 'feature_shuffling_corruption' to generate negative samples.
    # The model learns by maximizing Mutual Information between:
    #   - Positive: Real local patches + Global summary
    #   - Negative: Shuffled features + Global summary
    model = DeepGraphInfomax(
        hidden_channels=hidden_channels,
        encoder=encoder,
        summary=summary,
        corruption=feature_shuffling_corruption
    )
    
    # 5. Execute Training
    # Uses the differential learning rate strategy defined earlier.
    model = train_quantum_dgi(model, features, idx_edge, epochs=50)
    
    # 6. Extract Embeddings (Z)
    model.eval()
    with torch.no_grad():
        z, _, _ = model(features, idx_edge)
        
    return z




--> Feature Dimension (d): 16
--> Quantum Latent Space: 4 Qubits


In [7]:
z = run_experiment(data.x, data.edge_index)

--> Starting Training for 50 epochs...


Training: 100%|██████████| 50/50 [10:31<00:00, 12.62s/it]


In [9]:
from qgcn_lib.utils import perform_kmeans_clustering, visualize_embedding
from sklearn.metrics import normalized_mutual_info_score


# 1. Cluster Analysis (K-Means)
# We ask K-Means to find 3 clusters in the 4-dimensional quantum latent space
labels, z_np, score = perform_kmeans_clustering(z, 3)

# 2. Quantitative Metric (NMI)
# We compare the learned clusters against the true graph communities
# NMI = 1.0 means perfect correlation; NMI = 0.0 means random.
nmi = normalized_mutual_info_score(data.y.cpu().numpy(), labels)

print(f"--> Silhouette Score: {score:.4f}")
print(f"--> NMI Score (Cluster Quality): {nmi:.4f}")

# 3. Visualization (t-SNE)
# This generates a 2D projection of the quantum embeddings
visualize_embedding(z_np, labels, score, 3)

Silhouette Score (k=3): 0.4945
--> Silhouette Score: 0.4945
--> NMI Score (Cluster Quality): 0.8885
